# Air Quality Dataset Exploration

The air quality dataset contains the responses of a gas multisensor device deployed on the field in an Italian city. Hourly responses averages are recorded along with gas concentrations references from a certified analyzer (Vito et al., 2008).

Variable information: https://archive.ics.uci.edu/dataset/360/air+quality

**We will be looking at a dataset, and seeing what it contains to get familiar with the `numpy`, `matplotlib`, and `pandas` libraries.**

In [ ]:
# ! pip3 install matplotlib # to plot artifacts
# ! pip3 install numpy # to perform statistical functions like mean
# ! pip3 install pandas # to handle the dataset

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Viewing a Dataset

In [ ]:
df = pd.read_csv('AirQualityUCI.csv', delimiter=';', decimal=',')

To see our dataset, we can use `display(df)`

In [ ]:
display(df)

Another way to see our dataset is, `df.head()` which also takes a variable number of rows to print as input.

Try printing the first 5 and 10 entries with `df.head()`

In [ ]:
df.head(???)

In [ ]:
df.head(???)

As we look at this dataset, we see that we have some Unnamed columns with NaN entries. These empty columns do not provide use to us, so we need to drop these columns.

In [ ]:
df.dropna(how='all', axis=1, inplace=True) # remove NaN columns

display(df)


We have a lot of featuers and information in this data. Let's find out it's size so know how many features we have and how many data points we have.

In [ ]:
print(df.shape)

df.shape tells us the length and width of our dataset, which is also the amount of data we have vs. the number of features. How many datapoints and features do we have?

# Exploring the relationship between features and the target

So, now we know the features we have and the size of our dataset
 
Sometimes, you will have features that you know are not useful for your task. To remove these, you can use `df.drop()`

Let's practice using this method by dropping 'Time' and 'Date'.

Try dropping them by yourselves using `df.drop()`, which takes in a list of columns to drop and the axis as input.

In [ ]:
df = df.drop(??, axis=??)

df.head()

Now we can get started on seeing the correlation between our features!

Before we compute any statistics, there's one more cleanup step. This dataset uses a special sentinel value, **-200**, to mark sensor readings that are missing (this is documented on the UCI dataset page). If we leave -200 in the data, pandas and numpy will treat it as a real measurement, which will throw off our means, medians, and plots.

Let's replace every -200 with `NaN` so it's properly treated as missing data.

In [ ]:
df = df.replace(-200, np.nan)

df.head()

## Getting Basic Statistics and Pair Plot

### Getting Basic Statistics for Numerical Features

First, let's get the basic statistics we discussed in the slideshow for our numerical first feature, CO(GT).

In [ ]:
# Mean
mean = np.nanmean(df['CO(GT)'])

# Median
median = np.nanmedian(df['CO(GT)'])

# Standard Deviation
std = np.nanstd(df['CO(GT)'])

# Min
min = np.nanmin(df['CO(GT)'])

# Max
max = np.nanmax(df['CO(GT)'])

print('Basic Stats for CO(GT)')
print(f'Mean:\t{mean}')
print(f'Median:\t{median}')
print(f'Std:\t{std}')
print(f'Min:\t{min}')
print(f'Max:\t{max}')

Do the same for our second integer feature NMHC(GT).

In [ ]:
# Mean
mean = ??

# Median
median = ??

# Standard Deviation
std = ??

# Min
min = ??

# Max
max = ??

print('Basic Stats for NMHC(GT)')
print(f'Mean:\t{mean}')
print(f'Median:\t{median}')
print(f'Std:\t{std}')
print(f'Min:\t{min}')
print(f'Max:\t{max}')

Notice how different this looks from `CO(GT)`, most of `NMHC(GT)`'s values were the -200 sentinel, so after cleaning, a large portion of this column is now `NaN`. This is a common real-world issue: some sensors/features are much less reliable than others, and you may end up excluding a feature entirely if too much of it is missing.

### Making a Histograms
Let's look at the distribution of CO(GT) and see how well our statistics make sense.

In [ ]:
plt.hist(df['CO(GT)'], bins=10)
plt.title('CO(GT) Data Distribution')
plt.xlabel('Bins')
plt.ylabel('Frequency')
plt.show()

Do the same for NMHC(GT)! Try changing the bin size.

In [ ]:
???

### Pair Plot

What we want to know is what is the best feature to predict temperature?

So, we need to plot each feature against temperature, or `df['T']`. But eyeballing 12 scatter plots can be misleading, especially when there are thousands of overlapping points. To make this rigorous, we'll also compute the **Pearson correlation coefficient** (`r`) between each feature and `T`. `r` ranges from -1 to 1: values near 0 mean little to no linear relationship, and values close to -1 or 1 mean a strong negative or positive linear relationship.

We'll sort the features by how strongly they correlate with temperature, and plot the strongest relationships first.

In [ ]:
# Correlation of every feature with Temperature (T), sorted by strength (ignoring NaNs automatically)
correlations = df.drop(columns='T').corrwith(df['T']).sort_values(key=abs, ascending=False)

print(correlations)

In [ ]:
# Plot features in order of correlation strength, strongest first
features_sorted = correlations.index

fig, axes = plt.subplots(
    nrows=4,
    ncols=3,
    figsize=(15, 16)
)

for ax, feature in zip(axes.ravel(), features_sorted):
    r = correlations[feature]

    # Lower alpha and marker size since we have thousands of overlapping points
    ax.scatter(df[feature], df['T'], alpha=0.15, s=6)

    # Add a trend line so the relationship is visible through the noise
    valid = df[[feature, 'T']].dropna()
    coeffs = np.polyfit(valid[feature], valid['T'], 1)
    x_line = np.linspace(valid[feature].min(), valid[feature].max(), 100)
    ax.plot(x_line, np.polyval(coeffs, x_line), color='red', linewidth=1.5)

    ax.set_xlabel(feature)
    ax.set_ylabel('Temperature (T)')
    ax.set_title(f'{feature} vs. T  (r = {round(r, 2)})')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**What do we see?** `AH` (absolute humidity) has the strongest relationship with temperature (r $\approx$ 0.66), followed by `RH` (relative humidity, r $\approx$ -0.58) and `PT08.S4(NO2)` (r $\approx$ 0.56). This matches physical intuition, humidity and temperature are closely linked. Meanwhile, `CO(GT)`, `PT08.S5(O3)`, and `PT08.S1(CO)` barely correlate with temperature at all (r close to 0), and their trend lines are nearly flat.

One caution: `NMHC(GT)` shows a moderate r ($\approx$ 0.39), but remember from earlier that about 90% of its values were missing. A correlation computed from the remaining 10% of the data is much less reliable, so we shouldn't put too much trust in that number.

Based on this, **`AH` looks like our best single feature for predicting temperature** so we'll use it in the next section!

# References

Vito, S. (2008). Air Quality [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C59K5F.